In [ ]:
import os
import json
from typing import TypedDict, List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END


# ---------------------------------------------------------
# Load environment variables
# ---------------------------------------------------------

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY is not set.")
MODEL = "gemma-4-31b-it"
# ---------------------------------------------------------
# Gemini model
# ---------------------------------------------------------

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2,
)

# ---------------------------------------------------------
# Output schema
# ---------------------------------------------------------

class KeywordStrategy(BaseModel):
    topic: str
    search_intent: str = Field(
        description="Primary search intent of the topic"
    )
    primary_keywords: List[str] = Field(
        description="Main focus keywords for the blog"
    )
    secondary_keywords: List[str] = Field(
        description="Supporting SEO keywords"
    )
    long_tail_keywords: List[str] = Field(
        description="Long-tail search phrases"
    )
    question_keywords: List[str] = Field(
        description="Questions users may search"
    )
    semantic_keywords: List[str] = Field(
        description="Related entities, concepts and terminology"
    )
    blog_title_ideas: List[str] = Field(
        description="SEO-friendly blog title ideas"
    )
    content_angles: List[str] = Field(
        description="Different content angles for this topic"
    )

# ---------------------------------------------------------
# Structured Gemini model
# ---------------------------------------------------------

structured_llm = llm.with_structured_output(KeywordStrategy)

# ---------------------------------------------------------
# LangGraph State
# ---------------------------------------------------------

class BlogSEOState(TypedDict):

    trending_search: str
    keyword_strategy: dict

# ---------------------------------------------------------
# Node 1: Generate SEO keyword strategy
# ---------------------------------------------------------

def generate_keywords(state: BlogSEOState):
    print("inside generate keyword")
    trending_search = state["trending_search"]
    prompt = f"""
You are an expert SEO keyword strategist.
A trending search query has been detected:
TRENDING SEARCH:
{trending_search}
Your task is to create a keyword strategy that can be used
to generate SEO-focused blog articles.
Requirements:
1. Identify the main search intent.
2. Generate primary keywords.
   These should be strong candidates for the main focus keyword
   of an article.
3. Generate secondary keywords.
   These should support the main topic.
4. Generate long-tail keywords.
   These should represent specific searches users might perform.
5. Generate question keywords.
   These should resemble natural questions people could search.
6. Generate semantic keywords.
   Include closely related concepts, entities, terminology,
   subtopics and phrases.
7. Generate SEO-friendly blog title ideas.
8. Generate different content angles.
Important rules:
- Do NOT invent search volume.
- Do NOT invent keyword difficulty.
- Do NOT invent CPC.
- Do NOT claim that a keyword is actually trending unless
  that information was provided.
- Focus on semantic relevance and search intent.
- Avoid keyword stuffing.
- Avoid duplicate keywords.
- Make the keywords useful for an actual blog-writing pipeline.

Return the result according to the provided JSON schema.
"""

    result = structured_llm.invoke(prompt)
    return {
        "keyword_strategy": result.model_dump()
    }

# ---------------------------------------------------------
# Node 2: Validate / clean output
# ---------------------------------------------------------

def validate_keywords(state: BlogSEOState):
    print("inside validate")
    strategy = state["keyword_strategy"]
    # Remove duplicate keywords while preserving order
    for field in [
        "primary_keywords",
        "secondary_keywords",
        "long_tail_keywords",
        "question_keywords",
        "semantic_keywords",
        "blog_title_ideas",
        "content_angles",
    ]:

        if field in strategy:
            seen = set()
            cleaned = []
            for item in strategy[field]:
                normalized = item.strip().lower()
                if normalized not in seen:
                    seen.add(normalized)
                    cleaned.append(item.strip())
            strategy[field] = cleaned
    return {
        "keyword_strategy": strategy
    }

# ---------------------------------------------------------
# Build LangGraph
# ---------------------------------------------------------

graph = StateGraph(BlogSEOState)
graph.add_node(
    "generate_keywords",
    generate_keywords
)
graph.add_node(
    "validate_keywords",
    validate_keywords
)
graph.set_entry_point("generate_keywords")
graph.add_edge(
    "generate_keywords",
    "validate_keywords"
)
graph.add_edge(
    "validate_keywords",
    END
)
app = graph.compile()

# ---------------------------------------------------------
# Run

trending_search = "digital product engineering services india"

result = app.invoke({
        "trending_search": trending_search
    })

print(
        json.dumps(
            result["keyword_strategy"],
            indent=2,
            ensure_ascii=False
        )
    )

In [ ]:

from typing import List
from pydantic import BaseModel
from google import genai
from google.genai import types as google_types
import os
from dotenv import load_dotenv

load_dotenv()

# SEARCH_MODEL = "models/gemma-4-26b-a4b-it"
SEARCH_MODEL = "models/gemma-4-31b-it"

def _grounded_search(query: str, max_results: int = 8) -> List[dict]:
    print(1)
    api_key = os.getenv("GOOGLE_API_KEY")
    print("in function")
    if not api_key:
        return []

    try:
        print(2)
        client = genai.Client(api_key=api_key)
        print(3)
        print(" in try block")
        prompt = f"""
want to post a blog from embarkingonvoyage.com website can you give me top searched keywords/ queries in India where decision makers like CTOs, VPs, Digital heads might search to get a company like us ?
max results {max_results}
"""

        response = client.models.generate_content(
            model=SEARCH_MODEL,
            contents=prompt,
            config=google_types.GenerateContentConfig(
                tools=[
                    google_types.Tool(
                        google_search=google_types.GoogleSearch()
                    )
                ],
                response_mime_type="application/json",
            ),
        )
        print("after calling model")
        print("response:::::::::",response)
        if not response.parsed:
            print("[warning] Could not parse structured search response")
            return []
        # print(response)
        return [
            result.model_dump()
            for result in response.parsed.results
        ]

    except Exception as exc:
        print(f"[warning] grounded search failed: {exc}")
        return []

results = _grounded_search(
    "digital product engineering services india"
)

for result in results:
    print(result)

In [ ]:
from typing import List
from pydantic import BaseModel
from google import genai
from google.genai import types as google_types
import os
from dotenv import load_dotenv

load_dotenv()
api_key=os.getenv("GOOGLE_API_KEY")

# The SDK automatically picks up the GEMINI_API_KEY environment variable
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemma-4-31b-it", 
    contents="""steps:
    1. first identify services of embarkingonvoyage.com by querying
    2. draft keywords based on the serives found
    3. generate queries to find current (today is 10 september 2026, strictly include the date in every query you use) top searched keywords/ queries in india where decision makers like CTOs, VPs, Digital heads might search to get a company like us (embarkingonvoyage)
    4. search internet using the queries and give resulting queries and keywords"""
)
# prompt="don't give answers based on memory, search queries to get the desired result for this query:want to post a blog from embarkingonvoyage.com website can you give me top searched keywords/ queries in India where decision makers like CTOs, VPs, Digital heads might search to get a company like us ? max results 8"
print(response)

In [ ]:
!pip install serpapi

In [ ]:
import requests
import os
from dotenv import load_dotenv
load_dotenv()
from backend.variables.prompts import TOP_KEYWORD_QUERY, GATHER_LINKS_QUERY
def search_top_keywords(query):
    SERP_API_KEY=os.getenv("SERP_API_KEY")
    url = "https://serpapi.com/search"
    params = {
        "engine": "google_ai_mode",
        "q": query,
        "api_key": SERP_API_KEY,
        # India-focused search
        "location": "India",
        "gl": "in",

        # English-language decision-maker searches
        "hl": "en",

        # Structured response for your LangGraph/parser
        "output": "json",

        # Fresh research rather than cached result
        "no_cache": True,
    }
    
    try:
            # Fire standard HTTPS web call directly to the engine
        response = requests.get(url, params=params)
            
        if response.status_code != 200:
            print(f"❌ SerpApi server rejected query. Code: {response.status_code}")
            print(f"Server message: {response.text}")
            return []
        results = response.json()
    except Exception as e:
        print(f"Workflow execution pipeline failed: {e}")
        return []
    # print("result:::::::", results)
    text_blocks = results["text_blocks"]
    # print("=============================\ntext:::::::::::",text_blocks)
    return text_blocks

result=search_top_keywords(TOP_KEYWORD_QUERY)
print(type(result))

In [ ]:
import json
print(type(result[0]))
result1=json.dumps(result, indent=4)
result2=result[0]
print(type(result2))
print(result2)
result3=result2.get("code")
print(result3)

In [ ]:
import requests
import os
from dotenv import load_dotenv
load_dotenv()
from backend.variables.prompts import TOP_KEYWORD_QUERY, GATHER_LINKS_QUERY
def gather_links(query):
    SERP_API_KEY=os.getenv("SERP_API_KEY")
    url = "https://serpapi.com/search"
    params = {
        "engine": "google_ai_mode",
        "q": query,
        "api_key": SERP_API_KEY,
        # India-focused search
        "location": "India",
        "gl": "in",

        # English-language decision-maker searches
        "hl": "en",

        # Structured response for your LangGraph/parser
        "output": "json",

        # Fresh research rather than cached result
        "no_cache": True,
    }
    
    try:
            # Fire standard HTTPS web call directly to the engine
        response = requests.get(url, params=params)
            
        if response.status_code != 200:
            print(f"❌ SerpApi server rejected query. Code: {response.status_code}")
            print(f"Server message: {response.text}")
            return []
        results = response.json()
    except Exception as e:
        print(f"Workflow execution pipeline failed: {e}")
        return []
    # print("result:::::::", results)
    text_blocks = results["text_blocks"]
    # print("=============================\ntext:::::::::::",text_blocks)
    return text_blocks

result_links=gather_links(GATHER_LINKS_QUERY)

In [ ]:
print(result_links[0].get("code"))
print(type(result_links))

In [ ]:
from backend.tools import search_top_keywords
trending_keywords=search_top_keywords()
print(trending_keywords)

In [ ]:
import json

import json_repair
print(trending_keywords)
x=trending_keywords.get("research_summary")
print(x)
print(type(x))

In [ ]:
from backend.tools import gather_links
result_links=gather_links("agentic ai")
print(result_links)

In [ ]:
import json_repair
x='''{
"topic": "Enterprise Agentic AI Architecture Implementation",
"research_date": "2026-09-11",
"internal_links": [
{
"url": "https://embarkingonvoyage.com/services/ainative-digital-product-engineering/",
"title": "AI-Native Digital Product Engineering Services | EOV",
"page_type": "service",
"relevance_score": 98,
"why_relevant": "Directly covers engineering scalable cloud architectures and AI-augmented software required for transitioning agentic systems from experimental phases into enterprise production.",
"recommended_article_section": "Introduction & Enterprise Engineering Foundations",
"suggested_anchor_text": "AI-Native Digital Product Engineering Services",
"linking_purpose": "service_context"
},
{
"url": "https://embarkingonvoyage.com/blogs/best-ai-product-engineering-company-in-india/",
"title": "Best AI Product Engineering Company in India | AI-Native Apps",
"page_type": "blog",
"relevance_score": 95,
"why_relevant": "Outlines a structured 4-week framework covering multi-agent orchestration, context engineering, and guardrail integration which directly mirrors enterprise implementation steps.",
"recommended_article_section": "Multi-Agent Orchestration & Blueprinting Framework",
"suggested_anchor_text": "structured 4-Week AI MVP Framework",
"linking_purpose": "deeper_explanation"
},
{
"url": "https://embarkingonvoyage.com/data-engineering/why-data-engineering-for-enterprises-is-the-backbone-of-modern-businesses/",
"title": "Why Data Engineering for Enterprises is the Backbone of Modern Businesses?",
"page_type": "blog",
"relevance_score": 90,
"why_relevant": "Explains foundational enterprise data pipelines, data warehousing, and real-time data streaming necessary to feed context-aware autonomous agents.",
"recommended_article_section": "Data Pipelines & Context Engineering Layer",
"suggested_anchor_text": "enterprise data pipelines",
"linking_purpose": "supporting_example"
},
{
"url": "https://embarkingonvoyage.com",
"title": "Micro Services in Product Modernization",
"page_type": "technology",
"relevance_score": 85,
"why_relevant": "Details microservices architecture, RESTful API integrations, and CI/CD pipelines which form the decoupled execution backplane for agentic tool use and API orchestration.",
"recommended_article_section": "Action Layer & API Tool Integration Architecture",
"suggested_anchor_text": "Micro services based product modernization",
"linking_purpose": "deeper_explanation"
},
{
"url": "https://embarkingonvoyage.com",
"title": "Artificial General Intelligence: The Definitive Guide to AGI",
"page_type": "blog",
"relevance_score": 80,
"why_relevant": "Provides structural definitions distinguishing narrow automation from autonomous self-correcting cognitive loops essential for framing enterprise agent capabilities.",
"recommended_article_section": "Defining Agentic Capabilities vs Narrow AI",
"suggested_anchor_text": "evolutionary tiers of artificial intelligence",
"linking_purpose": "deeper_explanation"
}
],
"external_links": [
{
"url": "https://nist.gov",
"title": "Artificial Intelligence Risk Management Framework (AI RMF)",
"publisher": "National Institute of Standards and Technology (NIST)",
"source_type": "standards",
"authority_score": 99,
"primary_or_secondary": "primary",
"claim_supported": "Governing autonomous AI risks, trustworthiness characteristics, and validation controls for enterprise systems.",
"recommended_article_section": "Security, Governance & Guardrail Integration",
"suggested_anchor_text": "NIST AI Risk Management Framework",
"why_authoritative": "Official US government standards body providing universally accepted benchmarks for trustworthy AI deployment.",
"publication_or_update_date": "2023-01-27",
"current_verification_required": false
},
{
"url": "https://gartner.com",
"title": "What Is Agentic AI?",
"publisher": "Gartner",
"source_type": "consulting",
"authority_score": 95,
"primary_or_secondary": "secondary",
"claim_supported": "Market definition and enterprise strategic adoption timeline for autonomous agentic systems capable of independent planning.",
"recommended_article_section": "Executive Summary & Market Context",
"suggested_anchor_text": "Gartner analysis on Agentic AI",
"why_authoritative": "Preeminent global research and advisory firm tracking enterprise technology adoption metrics.",
"publication_or_update_date": "2024-10-01",
"current_verification_required": true
},
{
"url": "https://mckinsey.com",
"title": "The state of AI in early 2024: Gen AI adoption hinges and scaling challenges",
"publisher": "McKinsey & Company",
"source_type": "consulting",
"authority_score": 96,
"primary_or_secondary": "primary",
"claim_supported": "Enterprise scaling friction, cost metrics, and infrastructure hurdles when moving beyond isolated pilots.",
"recommended_article_section": "Enterprise Scaling & Operational Bottlenecks",
"suggested_anchor_text": "McKinsey Global Survey on AI adoption",
"why_authoritative": "Top-tier global management consulting firm conducting rigorous recurring empirical studies on enterprise tech integration.",
"publication_or_update_date": "2024-05-30",
"current_verification_required": true
},
{
"url": "https://nist.gov",
"title": "Secure Software Development Framework (SSDF) Version 1.1",
"publisher": "NIST",
"source_type": "government",
"authority_score": 98,
"primary_or_secondary": "primary",
"claim_supported": "Mitigating software supply chain vulnerabilities introduced by autonomous code execution and dynamic tool invocation.",
"recommended_article_section": "Action Layer Security & Sandboxing",
"suggested_anchor_text": "NIST Secure Software Development Framework",
"why_authoritative": "Gold standard federal cybersecurity guidance for secure software engineering life cycles.",
"publication_or_update_date": "2022-02-03",
"current_verification_required": false
}
],
"citation_targets": [
{
"claim_type": "market_claim",
"claim_to_verify": "A significant percentage of enterprise organizations plan to embed agentic AI workflows into core production applications within the next 24 months.",
"best_source_url": "https://gartner.com",
"citation_reason": "Provides executive-level analyst validation regarding market direction and enterprise deployment priority."
},
{
"claim_type": "security_claim",
"claim_to_verify": "Autonomous agents executing tool calls require rigorous runtime sandboxing and parameter validation to prevent prompt injection and unauthorized system modification.",
"best_source_url": "https://nist.gov",
"citation_reason": "Applies established secure software design principles to the novel threat vectors of agentic action layers."
}
],
"linking_notes": {
"internal_linking_summary": "Internal links strategically target EOV's core service offerings and granular technical sub-blogs, creating a logical path from high-level digital transformation to tactical execution frameworks (data engineering, microservices, and multi-agent MVP blueprints).",
"external_citation_summary": "External citations focus on high-authority regulatory/consulting pillars (NIST and Gartner/McKinsey) to back up operational risk controls and strategic market sizing without resorting to content-farm links.",
"missing_evidence": [
"A granular empirical benchmark study quantifying exact token-to-latency performance degradation across multi-vendor multi-agent loops at 100k+ concurrent enterprise requests."
]
}
}'''

print(type(x))
y=json_repair.loads(x)
print(y)
print(type(y))